In [3]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

## 1.初始化并调用模型

langchain提供了两种常见方法用来初始化模型：
- 使用init_chat_model方法，由langchain自动创建模型对象
- 使用不同模型对应的Model类，手动创建模型对象


### 1.1.init_chat_model
使用init_chat_model方法，langchain根据模型名称自动初始化与模型的连接，非常方便。

但需要注意的是：**如果使用我们必须在.env中配置好模型提供者的api_key**


In [4]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [5]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 1.2.自定义模型

init_chat_model默认会根据模型名称自动确定模型的提供者、其base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [6]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [7]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.3.调整模型参数
除了修改模型提供者以外，init_chat_model方法允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [8]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5,
    top_p=1.0,
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.4.使用model类
其实init_chat_model方法底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [9]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [10]:
# 打印结果
print(type(model))

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


# 2.访问模型

LangChain提供了两个不同的方法来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 2.1.invoke
invoke方法是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [11]:
# 通过invoke方法访问模型，需要阻塞等待模型生成结果
response = model.invoke("月亮的首都是哪里？")

In [12]:
# 查看响应内容
print(response)

content='月亮是地球的自然卫星，并没有首都。首都是指一个国家或地区的行政中心，而月球上并不存在这样的行政区划。如果您是在以一种比喻或者诗意的方式来提问，可能需要提供更多的上下文以便我能更好地理解您的意思。如果有关于月亮的具体科学问题或其他方面的问题，欢迎您提问！' additional_kwargs={} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': 'bed73a79-7daa-4616-8c6e-8bd12829db59', 'token_usage': {'input_tokens': 14, 'output_tokens': 67, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 81}} id='lc_run--019d3fb3-92ff-7261-91ba-8cc25220bbf5-0' tool_calls=[] invalid_tool_calls=[]


## 2.2.流式访问

阻塞式调用需要等待较长时间才能看到AI返回的结果，而流式调用则可以实时看到AI返回的一个个词。

In [13]:
# 通过.stream方法实现流式访问
stream = model.stream("月亮的首都是哪里？")

# 流式访问会返回一个生成器对象，可以遍历生成器对象来实时获取AI的回复
print(type(stream))

<class 'generator'>


In [14]:
# 遍历stream结果，实时打印AI的回复
for chunk in stream:
    print(chunk.content, end="", flush=True)

月亮是地球的自然卫星，并没有首都。首都是指一个国家的行政中心，而月亮上并不存在国家或城市的概念。不过，人类曾经登陆过月球，阿波罗11号任务中，尼尔·阿姆斯特朗和巴兹·奥尔德林成为了首次踏上月球表面的人类，但他们并没有在月球上建立任何形式的定居点或行政中心。如果您有关于太空探索或其他主题的问题，欢迎继续提问！

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent方法用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [15]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [16]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [17]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "月亮的首都是哪里？"}]
})

print(response)

{'messages': [HumanMessage(content='月亮的首都是哪里？', additional_kwargs={}, response_metadata={}, id='db4be584-1905-436a-ab94-d6b61a1efa61'), AIMessage(content='这是一个非常有趣且富有诗意的问题！从科学和现实的角度来说，月亮（月球）上没有城市，因此也没有所谓的“首都”。\n\n但是，如果我们跳出科学的框架，从**文化、传说和人类情感**的角度来看，这个问题可以有几种非常迷人的“答案”：\n\n### 1. 神话与传说中的“首都”\n*   **广寒宫**：在中国神话中，月亮上是嫦娥居住的宫殿，被称为“广寒宫”。这可以说是中国文化里月亮上最著名的“建筑”，如果月亮有首都，那很可能就是这里。\n*   **月球王国**：在许多西方童话和奇幻故事中，月亮有时被描绘成一个有国王、王后和子民的国度，自然也会有它的首都。但这属于纯粹的文学想象。\n\n### 2. 科幻作品中的“首都”\n在科幻题材里，人类在月球上建立基地或城市后，通常会有一个行政中心：\n*   **宁静海基地**：在许多科幻作品（如《2001太空漫游》）中，人类在月面“宁静海”区域建立大型基地，它常常扮演着月球活动中心的角色。\n*   **第谷城**：在其他科幻设定中，可能会以著名的环形山（如第谷环形山）命名主要城市。\n*   **阿波罗11号着陆点**：从历史意义上看，1969年阿波罗11号在**静海基地**的着陆点，是人类在月球上的第一个“落脚点”，具有无与伦比的象征意义，堪称月球上的“精神首都”。\n\n### 3. 一个哲学性的回答\n月亮的“首都”，也许就是它投射在**每个人心上的那一片光**。它是游子眼中的故乡，是诗人笔下的相思，是恋人之间的信物。在这个意义上，月亮的首都不在任何地理坐标上，而在人类共同的情感和文化记忆里。\n\n**总结一下：**\n*   **科学答案**：月球上没有首都。\n*   **人文答案**：**广寒宫**（中国神话）或**静海基地**（人类航天里程碑）是最接近“首都”概念的答案。\n*   **浪漫答案**：月亮的首都是“望月之人”的心里。\n\n所以，下次再有人问起，你可以根据场合，给出一个科学、浪漫或充满幻想的

### 流式访问


In [18]:
# 通过stream方法实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "月亮的首都是哪里？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [19]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

这是一个非常有趣且富有诗意的问题！从科学和现实的角度来说，**月亮（月球）本身没有首都，因为它不是一个国家或政治实体，也没有城市。**

但是，如果从**科幻、文学或幽默**的角度来解读，这个问题就打开了想象的大门。人们通常会用以下几种方式来“回答”：

### 1. **科幻/传说中的“首都”**
   * **静海基地**：在许多科幻作品（如《2001太空漫游》）和早期NASA的设想中，人类在月球上的第一个永久基地可能建在**静海**。阿波罗11号就着陆在这里，因此它具有历史象征意义，可以说是“精神首都”。
   * **柏拉图环形山**：在一些故事中，被设想为重要基地。
   * **广寒宫**：在中国神话中，月亮上有一座宫殿叫“广寒宫”，嫦娥和玉兔居住在那里。这可以说是中国传统文化中月亮的“首都”或中心。

### 2. **幽默/网络梗的回答**
   * **“月之都”**：在一些ACGN（动画、漫画、游戏、小说）作品中，比如《东方Project》，就存在一个虚构的“月之都”，居住着月之民。
   * **“坑爹呢！”**：这是一个常见的玩笑式回答，利用“月球表面都是坑（环形山）”的特点来调侃。

### 3. **现实中的“行政中心”设想**
   如果未来人类在月球建立殖民地并形成政治实体，那么它的首都很可能：
   * **第一个永久性、功能齐全的殖民城市**，可能位于月球南极（因为那里有连续光照区和水冰资源）。
   * **由主要开发国家或联盟共同商定命名**，也许会被叫做“新宁静市”、“阿尔忒弥斯市”或“万户市”等具有纪念意义的名字。

### **总结一下：**
*   **科学答案**：月球没有首都。
*   **人文诗意答案**：**广寒宫**（中国神话）。
*   **科幻经典答案**：**静海基地**。
*   **未来设想答案**：等待人类去建立。

所以，下次如果有人问你这个问题，你可以反问：“你问的是科幻里的，神话里的，还是未来的？” 这本身就成了一个充满创意的话题。